# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
# ML-07 Section 1 — connect to the FlyRank remote warehouse

import duckdb
import os
import pandas as pd
import numpy as np

# Create DuckDB connection
con = duckdb.connect()

# FlyRank warehouse
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

# Look for an existing Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("Hugging Face token found in environment.")

    # Create DuckDB Hugging Face secret
    con.execute(f"""
        CREATE OR REPLACE SECRET hf_secret (
            TYPE huggingface,
            TOKEN '{HF_TOKEN}'
        )
    """)
else:
    print("No HF_TOKEN environment variable found.")
    print("If the next query returns 401/403, add your Hugging Face READ token.")

# Test the warehouse
print("\nTesting FlyRank warehouse...")

test_query = f"""
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE}/**/*.parquet',
        hive_partitioning=true
    )
    LIMIT 5
"""

try:
    sample_df = con.execute(test_query).fetchdf()

    print("Warehouse connection successful.")
    print("\nSample data:")
    display(sample_df)

except Exception as e:
    print("\nWarehouse access failed.")
    print(type(e).__name__)
    print(str(e)[:2000])

No HF_TOKEN environment variable found.
If the next query returns 401/403, add your Hugging Face READ token.

Testing FlyRank warehouse...

Warehouse access failed.
HTTPException
HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_clients.parquet' (HTTP 401)


In [9]:
# Find the available warehouse parquet datasets

files = con.execute(f"""
    SELECT *
    FROM glob('{WAREHOUSE}/**/*.parquet')
""").fetchdf()

print("Parquet datasets/files found:", len(files))

display(files.head(50))

Parquet datasets/files found: 22


,file
0,hf://datasets/FlyRank/internship-warehouse/dim...
1,hf://datasets/FlyRank/internship-warehouse/dim...
2,hf://datasets/FlyRank/internship-warehouse/fac...
3,hf://datasets/FlyRank/internship-warehouse/fac...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [12]:
# ML-07 Section 1 — identify warehouse tables and schemas

import pandas as pd

files = con.execute(f"""
    SELECT file
    FROM glob('{WAREHOUSE}/**/*.parquet')
    ORDER BY file
""").fetchdf()

print("All warehouse Parquet files:")
for i, path in enumerate(files["file"], 1):
    print(f"{i:02d}. {path}")

print("\n" + "=" * 80)
print("Checking each dataset for schema and row count...")
print("=" * 80)

table_info = []

for path in files["file"]:
    try:
        info = con.execute(f"""
            SELECT
                COUNT(*) AS rows
            FROM read_parquet('{path}')
        """).fetchone()

        schema_tmp = con.execute(f"""
            DESCRIBE SELECT *
            FROM read_parquet('{path}')
        """).fetchdf()

        table_info.append({
            "file": path,
            "rows": info[0],
            "columns": ", ".join(schema_tmp["column_name"].tolist())
        })

    except Exception as e:
        table_info.append({
            "file": path,
            "rows": "ERROR",
            "columns": str(e)[:200]
        })

table_info_df = pd.DataFrame(table_info)

display(table_info_df)

All warehouse Parquet files:
01. hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
02. hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
03. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
04. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
05. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
06. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
07. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
08. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
09. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
10. hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025

,file,rows,columns
0,hf://datasets/FlyRank/internship-warehouse/dim...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
1,hf://datasets/FlyRank/internship-warehouse/dim...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
2,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
3,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
4,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
5,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
6,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
7,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
8,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...
9,hf://datasets/FlyRank/internship-warehouse/fac...,ERROR,HTTP Error: HTTP GET error on 'https://hugging...


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I rank items by **opportunity to improve search performance**: prioritize items that have enough impressions, relatively low observed CTR, and therefore appear to have room for improvement.

The baseline is **decision-support only**. A high score does not prove that an item should be changed; it identifies items that are worth human review.

### Reason codes

* **LOW_CTR** — observed CTR is below the comparison baseline.
* **HIGH_IMPRESSIONS** — the item has a relatively large number of impressions, so an improvement could affect more searches.
* **LOW_CTR_HIGH_VOLUME** — combines both low observed CTR and high impression volume.
* **LIMITED_DATA** — the item has relatively little impression data, so confidence is lower.
* **REVIEW** — the score identifies the item for review but does not prove that an action is correct.


In [13]:
# ML-07 Section 1 — authenticate DuckDB and inspect the real performance table

import duckdb
import pandas as pd
import numpy as np
import os

# Create/reuse DuckDB connection
con = duckdb.connect()

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

# ---------------------------------------------------------
# Get Hugging Face token from Google Colab Secrets
# ---------------------------------------------------------
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found.\n\n"
        "In Colab:\n"
        "1. Open the left sidebar.\n"
        "2. Click the key/Secrets icon.\n"
        "3. Add a secret named HF_TOKEN.\n"
        "4. Put your Hugging Face READ token there.\n"
        "5. Enable notebook access for the secret.\n"
        "6. Run this cell again."
    )

print("Hugging Face token detected.")

# ---------------------------------------------------------
# Give DuckDB access to Hugging Face
# ---------------------------------------------------------
con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB Hugging Face authentication configured.")

# ---------------------------------------------------------
# Actual ML-07 performance dataset
# ---------------------------------------------------------
PERF_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=*/data_0.parquet"
)

print("\nPerformance path:")
print(PERF_GLOB)

# ---------------------------------------------------------
# Inspect schema
# ---------------------------------------------------------
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )
""").fetchdf()

print("\nPerformance schema:")
display(schema)

# ---------------------------------------------------------
# Inspect sample rows
# ---------------------------------------------------------
sample_df = con.execute(f"""
    SELECT *
    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )
    LIMIT 10
""").fetchdf()

print("\nSample performance records:")
display(sample_df)


Hugging Face token detected.
DuckDB Hugging Face authentication configured.

Performance path:
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet

Performance schema:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



Sample performance records:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,True,True,True,False,21,0,1050,...,0,0,0,0,0,0,0,0,0,2025-01
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,True,True,True,False,13,0,127,...,0,0,0,0,0,0,0,0,0,2025-01
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,True,True,True,False,29,0,356,...,0,0,0,0,0,0,0,0,0,2025-01
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,True,True,True,False,5,0,103,...,0,0,0,0,0,0,0,0,0,2025-01
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,True,True,True,False,8,0,304,...,0,0,0,0,0,0,0,0,0,2025-01


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I built a simple baseline action score using the observed GSC performance from **April–June 2026**.

The score gives higher priority to content with **lower observed CTR and higher impression volume**. CTR is calculated as clicks divided by impressions, and the overall observed CTR is used as the comparison baseline.

The score is used only to **prioritize items for human review**. It is not a prediction of future performance or a guarantee that changing an item will improve results.

The ranked queue is sorted by action score and saved to:

`work/outputs/baseline_action_score.csv`


In [14]:
# ML-07 Section 2 — build ranked baseline action queue
# Uses observed GSC performance only.
# No future outcome, product flag, client name, or private query is used.

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Define the performance source
# ---------------------------------------------------------

PERF_GLOB = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=*/data_0.parquet"
)

# Use a fixed observed window.
# This prevents accidentally mixing in later/future observations.
START_MONTH = "2026-04"
END_MONTH = "2026-06"

print("Performance window:", START_MONTH, "to", END_MONTH)

# ---------------------------------------------------------
# 2. Aggregate content performance
# ---------------------------------------------------------

baseline = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(gsc_avg_position) AS avg_position

    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )

    WHERE month BETWEEN '{START_MONTH}' AND '{END_MONTH}'
      AND gsc_data_available = TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL

    GROUP BY
        client_hash_id,
        content_hash_id
""").fetchdf()

print("Candidate content rows:", len(baseline))
display(baseline.head())

Performance window: 2026-04 to 2026-06


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate content rows: 271046


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,509.0,1.0,0.001965,27.903753
1,client_62f4a7e64f5e0096,content_13a8105125458098,44.0,0.0,0.000000,6.471264
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,56.0,0.0,0.000000,4.477941
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,526.0,2.0,0.003802,9.699711
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,268.0,1.0,0.003731,26.435651


In [15]:
# ---------------------------------------------------------
# 3. Basic data checks
# ---------------------------------------------------------

assert (baseline["impressions"] >= 0).all(), \
    "Negative impressions detected."

assert (baseline["clicks"] >= 0).all(), \
    "Negative clicks detected."

assert (
    baseline["clicks"] <= baseline["impressions"]
).all(), \
    "Clicks cannot exceed impressions."

# Remove rows where CTR cannot be measured
baseline = baseline[
    baseline["impressions"] > 0
].copy()

# ---------------------------------------------------------
# 4. Establish simple observed baselines
# ---------------------------------------------------------

ctr_median = baseline["ctr"].median()
impressions_median = baseline["impressions"].median()

print(f"Observed median CTR: {ctr_median:.6f}")
print(f"Observed median impressions: {impressions_median:,.0f}")

# ---------------------------------------------------------
# 5. Create the action score
# ---------------------------------------------------------
#
# Higher score =
#   lower-than-median CTR
#   AND higher impression volume.
#
# The log transform prevents very large impression counts
# from completely dominating the score.
# ---------------------------------------------------------

baseline["ctr_gap"] = (
    ctr_median - baseline["ctr"]
).clip(lower=0)

baseline["volume_weight"] = np.log1p(
    baseline["impressions"]
)

baseline["action_score"] = (
    baseline["ctr_gap"] *
    baseline["volume_weight"]
)

# ---------------------------------------------------------
# 6. Reason codes
# ---------------------------------------------------------

baseline["reason_code"] = np.select(
    [
        (
            (baseline["ctr"] < ctr_median) &
            (baseline["impressions"] >= impressions_median)
        ),

        (
            (baseline["ctr"] < ctr_median) &
            (baseline["impressions"] < impressions_median)
        ),

        (
            (baseline["ctr"] >= ctr_median) &
            (baseline["impressions"] >= impressions_median)
        )
    ],
    [
        "LOW_CTR_HIGH_VOLUME",
        "LOW_CTR",
        "HIGH_VOLUME_REVIEW"
    ],
    default="REVIEW"
)

# ---------------------------------------------------------
# 7. Confidence note
# ---------------------------------------------------------

upper_volume = baseline["impressions"].quantile(0.75)

baseline["confidence_note"] = np.where(
    baseline["impressions"] >= upper_volume,
    "Higher confidence: high observed impression volume",
    np.where(
        baseline["impressions"] >= impressions_median,
        "Moderate confidence: observed volume is above median",
        "Lower confidence: limited observed volume"
    )
)

# ---------------------------------------------------------
# 8. Rank the complete queue
# ---------------------------------------------------------

baseline = baseline.sort_values(
    by=[
        "action_score",
        "impressions",
        "ctr"
    ],
    ascending=[
        False,
        False,
        True
    ]
).reset_index(drop=True)

baseline["rank"] = np.arange(
    1,
    len(baseline) + 1
)

# ---------------------------------------------------------
# 9. Final output columns
# ---------------------------------------------------------

queue_output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "action_score",
        "reason_code",
        "confidence_note"
    ]
].copy()

# ---------------------------------------------------------
# 10. Write required CSV
# ---------------------------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

queue_output.to_csv(
    output_path,
    index=False
)

# ---------------------------------------------------------
# 11. Final checks
# ---------------------------------------------------------

assert os.path.exists(output_path)

assert len(queue_output) > 0

assert queue_output["rank"].is_monotonic_increasing

assert queue_output["action_score"].notna().all()

print("\n" + "=" * 70)
print("ML-07 BASELINE QUEUE CREATED")
print("=" * 70)

print(f"Rows ranked       : {len(queue_output):,}")
print(f"Observed window   : {START_MONTH} to {END_MONTH}")
print(f"Median CTR        : {ctr_median:.6f}")
print(f"Median impressions: {impressions_median:,.0f}")
print(f"Output            : {output_path}")

print("\nTop 20 candidates:")
display(queue_output.head(20))

Observed median CTR: 0.000000
Observed median impressions: 170

ML-07 BASELINE QUEUE CREATED
Rows ranked       : 271,046
Observed window   : 2026-04 to 2026-06
Median CTR        : 0.000000
Median impressions: 170
Output            : work/outputs/baseline_action_score.csv

Top 20 candidates:


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,action_score,reason_code,confidence_note
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,1986586.0,15428.0,0.007766,2.279094,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
1,2,client_e547b89c05043229,content_545bb6cc7081ded3,992336.0,4670.0,0.004706,2.363121,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
2,3,client_e547b89c05043229,content_963de14b1f58978f,908095.0,2405.0,0.002648,6.146574,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
3,4,client_8ddc46da5414ffd8,content_943dc881428182b8,680046.0,914.0,0.001344,3.498362,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
4,5,client_62f4a7e64f5e0096,content_acbcc847f8996314,611708.0,767.0,0.001254,4.247548,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
5,6,client_e547b89c05043229,content_21309e9a83c83653,596157.0,1025.0,0.001719,4.982741,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
6,7,client_62f4a7e64f5e0096,content_f107e54b10b43725,574397.0,1544.0,0.002688,4.795604,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
7,8,client_e547b89c05043229,content_0e03de7680314cd5,565913.0,1439.0,0.002543,2.584560,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
8,9,client_73cda7b4e4f265ea,content_33d31496fca9665e,557202.0,125.0,0.000224,6.175898,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...
9,10,client_73cda7b4e4f265ea,content_62770e1299963fe4,528618.0,583.0,0.001103,5.101484,0.0,HIGH_VOLUME_REVIEW,Higher confidence: high observed impression vo...


In [21]:
# ML-07 diagnostic — check why action scores are zero

print("CTR median:", ctr_median)
print("CTR mean:", baseline["ctr"].mean())
print("Weighted CTR:",
      baseline["clicks"].sum() / baseline["impressions"].sum())

print("\nAction score summary:")
display(baseline["action_score"].describe())

print("\nReason-code counts:")
display(baseline["reason_code"].value_counts())

CTR median: 0.0
CTR mean: 0.005469792498246185
Weighted CTR: 0.003839716386743851

Action score summary:


,action_score
count,271046.000000
mean,0.010990
std,0.009077
min,0.000000
25%,0.002661
50%,0.010133
75%,0.018350
max,0.049094



Reason-code counts:


,count
reason_code,
LOW_CTR,119621
LOW_CTR_HIGH_VOLUME,96637
HIGH_VOLUME_REVIEW,38947
REVIEW,15841


In [19]:
# ---------------------------------------------------------
# ML-07 — corrected baseline action score
# ---------------------------------------------------------

# Overall observed CTR across the review window
baseline_ctr = (
    baseline["clicks"].sum()
    / baseline["impressions"].sum()
)

# Impression benchmark
impressions_median = baseline["impressions"].median()

print(f"Overall observed CTR: {baseline_ctr:.6f}")
print(f"Median impressions: {impressions_median:,.0f}")

# Opportunity = how far the item's CTR is below
# the overall observed CTR
baseline["ctr_gap"] = (
    baseline_ctr - baseline["ctr"]
).clip(lower=0)

# Log volume prevents huge impression counts
# from completely dominating the score
baseline["volume_weight"] = np.log1p(
    baseline["impressions"]
)

baseline["action_score"] = (
    baseline["ctr_gap"] *
    baseline["volume_weight"]
)

# Reason codes
baseline["reason_code"] = np.select(
    [
        (
            (baseline["ctr"] < baseline_ctr) &
            (baseline["impressions"] >= impressions_median)
        ),
        (
            (baseline["ctr"] < baseline_ctr) &
            (baseline["impressions"] < impressions_median)
        ),
        (
            (baseline["ctr"] >= baseline_ctr) &
            (baseline["impressions"] >= impressions_median)
        )
    ],
    [
        "LOW_CTR_HIGH_VOLUME",
        "LOW_CTR",
        "HIGH_VOLUME_REVIEW"
    ],
    default="REVIEW"
)

# Confidence based on observed volume
upper_volume = baseline["impressions"].quantile(0.75)

baseline["confidence_note"] = np.where(
    baseline["impressions"] >= upper_volume,
    "Higher confidence: high observed impression volume",
    np.where(
        baseline["impressions"] >= impressions_median,
        "Moderate confidence: observed volume is above median",
        "Lower confidence: limited observed volume"
    )
)

# Rank again
baseline = baseline.sort_values(
    by=["action_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(
    1,
    len(baseline) + 1
)

# Rebuild output
queue_output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "action_score",
        "reason_code",
        "confidence_note"
    ]
].copy()

# Write corrected CSV
output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(
    os.path.dirname(output_path),
    exist_ok=True
)

queue_output.to_csv(
    output_path,
    index=False
)

print("\nCorrected baseline created.")
print("Output:", output_path)

print("\nTop 20:")
display(queue_output.head(20))

print("\nScore summary:")
display(queue_output["action_score"].describe())

Overall observed CTR: 0.003840
Median impressions: 170

Corrected baseline created.
Output: work/outputs/baseline_action_score.csv

Top 20:


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,action_score,reason_code,confidence_note
0,1,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,382947.0,8.0,0.000021,18.122212,0.049094,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
1,2,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,333712.0,2.0,0.000006,14.744692,0.048757,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
2,3,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,284085.0,1.0,0.000004,16.974464,0.048171,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
3,4,client_73cda7b4e4f265ea,content_33d31496fca9665e,557202.0,125.0,0.000224,6.175898,0.047834,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
4,5,client_a80fca3f171ed1de,content_012de75c008aa653,255868.0,1.0,0.000004,15.981706,0.047765,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
5,6,client_23a62021009f63c4,content_c60628276389acbb,285809.0,15.0,0.000052,51.990701,0.047579,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
6,7,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,396353.0,73.0,0.000184,7.087442,0.047120,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
7,8,client_73cda7b4e4f265ea,content_23a42776a7009b65,205433.0,1.0,0.000005,11.596141,0.046911,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
8,9,client_73cda7b4e4f265ea,content_8e1334d6356668e3,165689.0,0.0,0.000000,13.693177,0.046145,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,352484.0,89.0,0.000252,9.171847,0.045819,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...



Score summary:


,action_score
count,271046.000000
mean,0.010990
std,0.009077
min,0.000000
25%,0.002661
50%,0.010133
75%,0.018350
max,0.049094


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 items produced by the baseline action score.

The recommended action for each item is **human review**, not an automatic change. The reason code explains why the item entered the queue, while the confidence note reflects the amount of observed impression data.

A high score is treated as a review priority rather than proof that the content is underperforming. The recommendation could be wrong if the observed CTR is affected by legitimate search intent, ranking position, seasonality, or limited data.

For each candidate, I also recorded what could make the recommendation wrong so that the queue remains decision-support rather than an automatic optimization rule.


In [22]:
top20_review = queue_output.head(20).copy()

top20_review["action"] = "Human review"

top20_review["what_would_make_it_wrong"] = np.where(
    top20_review["impressions"] < impressions_median,
    "Limited observed volume may make CTR unstable.",
    "Low CTR may reflect legitimate intent, ranking position, seasonality, or other factors not captured by this baseline."
)

top20_review = top20_review[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
        "impressions",
        "clicks",
        "ctr",
        "action_score"
    ]
]

display(top20_review)

assert len(top20_review) == 20

,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong,impressions,clicks,ctr,action_score
0,1,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",382947.0,8.0,0.000021,0.049094
1,2,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",333712.0,2.0,0.000006,0.048757
2,3,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",284085.0,1.0,0.000004,0.048171
3,4,client_73cda7b4e4f265ea,content_33d31496fca9665e,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",557202.0,125.0,0.000224,0.047834
4,5,client_a80fca3f171ed1de,content_012de75c008aa653,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",255868.0,1.0,0.000004,0.047765
5,6,client_23a62021009f63c4,content_c60628276389acbb,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",285809.0,15.0,0.000052,0.047579
6,7,client_a80fca3f171ed1de,content_c1f764a2f362d1c3,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",396353.0,73.0,0.000184,0.047120
7,8,client_73cda7b4e4f265ea,content_23a42776a7009b65,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",205433.0,1.0,0.000005,0.046911
8,9,client_73cda7b4e4f265ea,content_8e1334d6356668e3,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",165689.0,0.0,0.000000,0.046145
9,10,client_a80fca3f171ed1de,content_9540d884af3e41fd,Human review,LOW_CTR_HIGH_VOLUME,Higher confidence: high observed impression vo...,"Low CTR may reflect legitimate intent, ranking...",352484.0,89.0,0.000252,0.045819


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline prioritizes items with low observed CTR and high impression volume. The top candidates are therefore review priorities, not automatic recommendations.

The weakest picks are candidates where the observed CTR may be unstable or where the low CTR may have a legitimate explanation. Possible reasons include search intent, ranking position, seasonality, or other factors not represented by this baseline.

For leakage, the score was calculated from observed GSC impressions and clicks in the fixed April–June 2026 review window. I did not use future outcomes, product flags, GA4 outcomes, AI referral fields, client names, URLs, or private queries in the score.

The result is decision-support only: it identifies items worth human review and does not claim that changing an item will definitely improve future performance.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 4 — Weak picks + leakage check

print("=" * 70)
print("WEAK PICKS")
print("=" * 70)

# Lower-confidence candidates:
# items below the median observed impression volume.
weak_picks = queue_output[
    queue_output["impressions"] < impressions_median
].sort_values(
    ["action_score", "impressions"],
    ascending=[True, True]
).head(10)

display(weak_picks)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

# 1. Confirm fixed observation window
print("Review window:", START_MONTH, "to", END_MONTH)

assert START_MONTH == "2026-04"
assert END_MONTH == "2026-06"

# 2. Confirm only intended performance fields were used
score_inputs = [
    "gsc_impressions",
    "gsc_clicks"
]

print("\nScore inputs:")
for col in score_inputs:
    print(" -", col)

# 3. Confirm no product/client/query fields are part of the score
forbidden_score_terms = [
    "product",
    "query",
    "url",
    "future",
    "outcome",
    "label",
    "target"
]

score_column_text = " ".join(
    queue_output.columns
).lower()

for term in forbidden_score_terms:
    assert term not in score_column_text, (
        f"Potential leakage term found in output columns: {term}"
    )

# 4. Confirm GA4 and AI referral fields were not used
unused_fields = [
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

print("\nExplicitly unused fields:")
for col in unused_fields:
    print(" -", col)

# 5. Recalculate CTR independently
check_ctr = np.where(
    queue_output["impressions"] > 0,
    queue_output["clicks"] / queue_output["impressions"],
    np.nan
)

np.testing.assert_allclose(
    queue_output["ctr"].to_numpy(),
    check_ctr,
    rtol=1e-10,
    atol=1e-12
)

# 6. Basic validity checks
assert (queue_output["impressions"] >= 0).all()
assert (queue_output["clicks"] >= 0).all()
assert (
    queue_output["clicks"]
    <= queue_output["impressions"]
).all()

assert queue_output["action_score"].notna().all()
assert queue_output["rank"].is_monotonic_increasing

# 7. Confirm required output exists
assert os.path.exists(
    "work/outputs/baseline_action_score.csv"
)

print("\n" + "=" * 70)
print("ALL ML-07 LEAKAGE / CONSISTENCY CHECKS PASSED")
print("=" * 70)

print("Output:", "work/outputs/baseline_action_score.csv")
print("Rows:", len(queue_output))
print("Top-20 rows:", len(queue_output.head(20)))

WEAK PICKS


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,action_score,reason_code,confidence_note
271018,271019,client_3ffa76342f366962,content_6324839bada7c8e2,1.0,1.0,1.0,3.0,0.0,REVIEW,Lower confidence: limited observed volume
271019,271020,client_3ffa76342f366962,content_64805b2bf53d11f5,1.0,1.0,1.0,0.0,0.0,REVIEW,Lower confidence: limited observed volume
271020,271021,client_3197e6291363b4db,content_d2c8213bf7085978,1.0,1.0,1.0,38.0,0.0,REVIEW,Lower confidence: limited observed volume
271021,271022,client_3ffa76342f366962,content_ec763081c8c525af,1.0,1.0,1.0,1.0,0.0,REVIEW,Lower confidence: limited observed volume
271022,271023,client_3ffa76342f366962,content_7dae3c3b40125257,1.0,1.0,1.0,24.0,0.0,REVIEW,Lower confidence: limited observed volume
271023,271024,client_3ffa76342f366962,content_0b91d051d93c2e59,1.0,1.0,1.0,2.0,0.0,REVIEW,Lower confidence: limited observed volume
271024,271025,client_3ffa76342f366962,content_55daed86ece6c09c,1.0,1.0,1.0,0.0,0.0,REVIEW,Lower confidence: limited observed volume
271025,271026,client_2b4306c3ed003f01,content_abc9db5c88daf61e,1.0,1.0,1.0,59.0,0.0,REVIEW,Lower confidence: limited observed volume
271026,271027,client_3ffa76342f366962,content_d06c1d5de5adc51e,1.0,1.0,1.0,1.0,0.0,REVIEW,Lower confidence: limited observed volume
271027,271028,client_d211cb07b9059bab,content_9d6c8a754441dd61,1.0,1.0,1.0,9.0,0.0,REVIEW,Lower confidence: limited observed volume



LEAKAGE CHECK
Review window: 2026-04 to 2026-06

Score inputs:
 - gsc_impressions
 - gsc_clicks

Explicitly unused fields:
 - ga4_pageviews
 - ga4_sessions
 - ga4_users
 - ga4_engaged_sessions
 - sessions_ai
 - ai_chatgpt
 - ai_perplexity
 - ai_gemini
 - ai_copilot
 - ai_claude
 - ai_meta
 - ai_other

ALL ML-07 LEAKAGE / CONSISTENCY CHECKS PASSED
Output: work/outputs/baseline_action_score.csv
Rows: 271046
Top-20 rows: 20


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.